# Performance and Scalability Tutorial

This tutorial demonstrates QuantStrata's performance features: backend selection (NumPy/Numba), parallel portfolio pricing, pricer result caching, and optional JAX MC pricing.

**Topics covered:**
- Benchmarking with the performance framework (Numba vs NumPy)
- Parallel portfolio pricing (sequential vs parallel, same result)
- Pricer result cache (timing with cache on vs off)
- Optional: JAX MC pricer for FX vanilla (when JAX is installed)

In [ ]:
import sys
sys.path.insert(0, '../../..')

import numpy as np
import time

print("Setup complete.")

## 1. Backend selection and benchmarking

Use the performance backend and benchmark framework to compare NumPy vs Numba (when available).

In [ ]:
from src.core.performance.backend import get_backend, get_backend_info, Backend
from src.core.performance.benchmark import benchmark_function, BenchmarkResult

info = get_backend_info()
print("Backend info:", info)
print("Selected backend:", get_backend("auto"))

In [ ]:
# Simple function to benchmark: vectorized dot product
def compute_dot(n: int):
    a = np.random.standard_normal(n).astype(np.float64)
    b = np.random.standard_normal(n).astype(np.float64)
    return float(np.dot(a, b))

result = benchmark_function(
    lambda: compute_dot(1_000_000),
    n_runs=5,
    n_warmup=1,
    name="dot_1M",
)
print(result)

## 2. Parallel portfolio pricing

Build a small portfolio and compare sequential vs parallel pricing (same result, optional speedup).

In [ ]:
from src.portfolio.portfolio import PortfolioPricer
from src.portfolio.parallel import ParallelPortfolioPricer
from src.portfolio.core import Portfolio, Position
from src.pricers.registry import DefaultPricerRegistry
from src.instruments.fx.options.vanilla import FxVanillaEuropeanOption
from src.marketdata.core.ids import MarketId
from src.marketdata.core.market import Market
from src.marketdata.core.interfaces import Quote
from src.marketdata.curves.term_structure import FlatZeroRateCurve
from src.marketdata.surfaces.vol_surface import FlatVolSurface

# Build registry and a simple market (minimal for demo)
reg = DefaultPricerRegistry().build()
spot_id = MarketId("FX", "SPOT", "EURUSD")
dom_id = MarketId("IR", "CURVE", "EUR.OIS")
fgn_id = MarketId("IR", "CURVE", "USD.OIS")
vol_id = MarketId("FX", "VOL", "EURUSD.VOL")
curve_d = FlatZeroRateCurve(continuously_compounded_rate=0.02)
curve_f = FlatZeroRateCurve(continuously_compounded_rate=0.01)
vol_surface = FlatVolSurface(sigma=0.15)
market = Market(
    asof="0.0",
    quotes={spot_id: Quote(value=1.10)},
    curves={dom_id: curve_d, fgn_id: curve_f},
    vols={vol_id: vol_surface},
)

# Small portfolio: a few FX vanilla options
opt = FxVanillaEuropeanOption(
    expiry=1.0,
    strike=1.10,
    option_type="call",
    notional=1_000_000.0,
    spot_id=spot_id,
    domestic_curve_id=dom_id,
    foreign_curve_id=fgn_id,
    vol_id=vol_id,
)
positions = [Position(instrument=opt, quantity=1.0)] * 5  # 5 same options for demo
portfolio = Portfolio(positions=positions)

base_pricer = PortfolioPricer(pricer_registry=reg)
parallel_pricer = ParallelPortfolioPricer(portfolio_pricer=base_pricer, max_workers=4)

seq_result = base_pricer.price(portfolio, market)
par_result = parallel_pricer.price(portfolio, market)

print("Sequential total PV:", seq_result.totals.pv)
print("Parallel total PV:", par_result.totals.pv)
print("Same result:", np.isclose(seq_result.totals.pv, par_result.totals.pv))

In [ ]:
# Optional: time sequential vs parallel (may show speedup for larger portfolios)
def time_pricer(pricer, port, mkt, n=10):
    start = time.perf_counter()
    for _ in range(n):
        pricer.price(port, mkt)
    return (time.perf_counter() - start) / n

t_seq = time_pricer(base_pricer, portfolio, market)
t_par = time_pricer(parallel_pricer, portfolio, market)
print(f"Sequential avg: {t_seq*1000:.2f} ms")
print(f"Parallel avg:   {t_par*1000:.2f} ms")

## 3. Pricer result cache

Wrap the portfolio pricer with a cache; repeated calls with the same portfolio and market return the cached result.

In [ ]:
from src.portfolio.caching import CachingPortfolioPricer

caching_pricer = CachingPortfolioPricer(
    portfolio_pricer=base_pricer,
    max_size=1024,
    ttl_seconds=None,
)

# First call: miss, compute and cache
r1 = caching_pricer.price(portfolio, market)
# Second call: hit, return cached
r2 = caching_pricer.price(portfolio, market)

print("First call PV:", r1.totals.pv)
print("Second call PV (cached):", r2.totals.pv)
print("Same:", r1.totals.pv == r2.totals.pv)

In [ ]:
# Timing: many repeated prices — cache should be much faster
n_repeat = 100
start = time.perf_counter()
for _ in range(n_repeat):
    caching_pricer.price(portfolio, market)
cached_time = (time.perf_counter() - start) / n_repeat

start = time.perf_counter()
for _ in range(n_repeat):
    base_pricer.price(portfolio, market)
uncached_time = (time.perf_counter() - start) / n_repeat

print(f"Uncached avg: {uncached_time*1000:.2f} ms")
print(f"Cached avg:   {cached_time*1000:.2f} ms")

## 4. Optional: JAX MC pricer (FX vanilla)

If JAX is installed, the JAX MC pricer is registered with `pricer_id="jax_mc"`. Compare FX vanilla price and timing vs BSM analytic or NumPy MC.

In [ ]:
from src.core.performance.backend import jax_available

if jax_available():
    # Resolve JAX MC pricer and BSM pricer
    jax_mc_pricer = reg.resolve(opt, pricer_id="jax_mc")
    bsm_pricer = reg.resolve(opt)  # default is BSM
    
    pv_bsm = bsm_pricer.price(opt, market)
    pv_jax = jax_mc_pricer.price(opt, market)
    
    print("BSM (analytic) PV:", pv_bsm)
    print("JAX MC PV:", pv_jax)
    print("Relative diff:", abs(pv_jax - pv_bsm) / (pv_bsm or 1))
    
    # Quick timing (JAX may be faster on GPU or large paths)
    n = 5
    start = time.perf_counter()
    for _ in range(n):
        jax_mc_pricer.price(opt, market)
    print(f"JAX MC avg ({n} runs): {(time.perf_counter()-start)/n*1000:.2f} ms")
else:
    print("JAX not installed. Install with: pip install jax jaxlib")